## Witaj w Labie 3, Tydzień 1 Dzień 4

Dziś zbudujemy coś z natychmiastową wartością! To początek labu, który potrwa 2 dni.

I zbudujemy ręcznie Agent Loop bez żadnego Agent Frameworka..

### Najpierw trochę przygotowań

W folderze `twin` umieściłem jeden plik `linkedin.pdf` - to pobrany PDF mojego profilu LinkedIn.

Zastąp go swoim! Powinieneś móc pobrać go ze swojego profilu LinkedIn; wejdź na stronę swojego profilu i użyj menu pod swoim nazwiskiem. Jeśli nie masz dostępu do tej funkcji, świetnie sprawdzi się dowolny PDF, np. Twoje CV.

Zrobiłem też plik o nazwie `summary.txt` w `twin` - przeczytaj go i zaktualizuj tak, żeby odzwierciedlał Ciebie.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Wyszukiwanie pakietów</h2>
            <span style="color:#00bfff;">W tym labie użyjemy wspaniałego pakietu Gradio do budowania szybkich UI, 
            a także popularnego czytnika PDF PyPDF. Jeśli zastanawiasz się, jak wybierać pakiety do swoich własnych projektów, zobacz Q37 na stronie <a href="https://edwarddonner.com/avatar?q=37">FAQ</a>.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Jeśli nie wiesz, co robi któryś z tych pakietów - zawsze możesz poprosić ChatGPT o wyjaśnienie!

from dotenv import load_dotenv
from anthropic import Anthropic
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)  # wczytuje .env i nadpisuje już ustawione zmienne środowiskowe
anthropic = Anthropic()  # klient automatycznie odczyta klucz z ANTHROPIC_API_KEY w .env

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## Dygresja: Trzy pojęcia jako przypomnienie

1. System Prompt: część danych wejściowych do LLM, która opisuje ogólny kontekst rozmowy

2. Historia konwersacji: cała dotychczasowa rozmowa

3. Iluzja pamięci: każda wiadomość do LLM jest bezstanowa. Przekazujemy całą dotychczasową rozmowę, żeby stworzyć iluzję, że model pamięta, co powiedziano 30 sekund temu...

__Więcej na ten temat w moim towarzyszącym kursie AI Engineer Core Track (pierwszy tydzień)__

In [ ]:
system_message = "Jesteś pomocnym asystentem"  # u Anthropic system prompt to osobny parametr, nie wpis w liście messages
messages = [
    {"role": "user", "content": "Cześć, mam na imię Ed"}
]

In [ ]:
response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages)  # max_tokens jest wymagany przez Anthropic API
print(next(block.text for block in response.content if block.type == "text"))  # response.content to lista bloków (thinking/text/...), wyciągamy pierwszy tekstowy

In [ ]:
system_message = "Jesteś zgryźliwym, błyskotliwym asystentem"
messages = [
    {"role": "user", "content": "Cześć, mam na imię Ed"}
]

In [ ]:
response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages)
print(next(block.text for block in response.content if block.type == "text"))

In [ ]:
system_message = "Jesteś zgryźliwym, błyskotliwym asystentem"
messages = [
    {"role": "user", "content": "Jak mam na imię?"}
]

In [ ]:
response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages)
print(next(block.text for block in response.content if block.type == "text"))

In [ ]:
system_message = "Jesteś zgryźliwym, błyskotliwym asystentem"
messages = [
    {"role": "user", "content": "Cześć, mam na imię Ed"},
    {"role": "assistant", "content": "Cześć, Ed. Miło Cię poznać."},
    {"role": "user", "content": "Jak mam na imię?"}
]

In [ ]:
response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_message, messages=messages)
print(next(block.text for block in response.content if block.type == "text"))

## Wracamy do głównego wątku!

Mamy profil LinkedIn w zmiennej `linkedin`

Mamy podsumowanie w zmiennej `summary`

Skonstruujmy System Prompt..

In [ ]:
system_prompt = f"""

# Twoja rola

Jesteś cyfrowym bliźniakiem działającym na stronie internetowej, rozmawiającym z jej odwiedzającymi.
Reprezentujesz osobę, do której należy ta strona.
Odpowiadasz na pytania dotyczące jej kariery, doświadczenia, umiejętności i historii zawodowej.

Oto szczegóły dotyczące osoby, którą reprezentujesz:

{summary}

Jeśli zostaniesz o to zapytany, wyjaśnij jasno, że jesteś AI będącym cyfrowym bliźniakiem tej osoby.

# Kontekst

Oto podsumowanie profilu LinkedIn tej osoby, dzięki któremu możesz odpowiadać na pytania:

{linkedin}

# Zasady

Angażuj się w rozmowę z użytkownikiem. Bądź profesjonalny i przystępny, jakbyś rozmawiał z potencjalnym klientem albo przyszłym pracodawcą, który trafił na tę stronę.
Unikaj odpowiadania na pytania niezwiązane z karierą, doświadczeniem, umiejętnościami i historią zawodową użytkownika;
sprowadzaj rozmowę z powrotem na tematy zawodowe.

Zawsze pozostawaj w roli cyfrowego bliźniaka osoby, którą reprezentujesz. Reprezentuj tę osobę.

WAŻNE: Jeśli nie znasz odpowiedzi, powiedz to wprost. Nigdy nie zmyślaj odpowiedzi.
Jeśli użytkownik zapyta o coś, czego nie ma w kontekście, powiedz, że tego nie wiesz.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "user", "content": "Cześć - opowiedz mi coś o sobie"}
]

In [ ]:
response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, messages=messages)
display(Markdown(next(block.text for block in response.content if block.type == "text")))

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]  # Gradio może dawać dodatkowe klucze w historii (np. metadata) - Anthropic akceptuje tylko role/content
    messages = history + [{"role": "user", "content": message}]
    response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, messages=messages)  # system_prompt jako top-level parametr, nie wpis w messages
    return next(block.text for block in response.content if block.type == "text")

In [ ]:
chat("Podsumuj, kim jesteś", [])

## UWAGA dla tych, którzy nie używają modeli OpenAI

Jeśli używasz modeli innych niż OpenAI, może być konieczne wstawienie tej linii na początku chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# A teraz - NARZĘDZIA!

Zacznijmy od funkcji...

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

## Krok 1 - napisz json opisujący narzędzie


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Użyj tego narzędzia, żeby zapisać adres email podany przez użytkownika",  # opis czytany przez model przy decyzji, czy wywołać narzędzie
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "Adres email tego użytkownika"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [record_email_tool_json]  # Anthropic przyjmuje płaską listę definicji narzędzi, bez opakowania {"type": "function", "function": ...} jak w OpenAI

In [ ]:
tools

## Krok 2 - nowa funkcja chat()

Tutaj implementujemy wywołanie narzędzia.

W rzeczywistości jest to trochę toporne. To jak zobaczenie składników wykwintnego przepisu i odkrycie, że te składniki są całkiem zwyczajne.

Wywoływanie narzędzi to instrukcja "if". W tym przypadku zakodowaliśmy na sztywno wszystko, zakładając, że jedynym narzędziem jest narzędzie do maili.

DYGRESJA: Jeśli myślisz - ale czekaj! Powinienem to zapamiętać, żeby móc zrobić to sam! To kluczowy punkt jest taki: to właśnie tym zajmują się za Ciebie Agent Frameworki. W praktyce prawdopodobnie nigdy więcej sam tego nie napiszesz. Jesteśmy osłonięci przed tymi instrukcjami if przez Agent Framework. Dlatego często są one opisywane jako "warstwy abstrakcji".

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]  # Gradio może dawać dodatkowe klucze w historii - Anthropic akceptuje tylko role/content
    messages = history + [{"role": "user", "content": message}]
    response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, tools=tools, messages=messages)

    if response.stop_reason == "tool_use":  # Anthropic sygnalizuje chęć użycia narzędzia przez stop_reason, nie finish_reason jak OpenAI
            tool_use = next(block for block in response.content if block.type == "tool_use")  # pierwszy (i tu jedyny) blok tool_use w odpowiedzi
            email = tool_use.input.get("email")  # input jest już sparsowanym dict, nie JSON-stringiem jak tool_call.function.arguments w OpenAI
            record_email_tool(email)
            messages.append({"role": "assistant", "content": response.content})  # cała odpowiedź assistant (razem z blokiem tool_use) wraca do historii
            messages.append({
                "role": "user",  # wynik narzędzia wraca jako wiadomość user z blokiem tool_result, nie rola "tool" jak w OpenAI
                "content": [{
                    "type": "tool_result",
                    "tool_use_id": tool_use.id,  # musi się zgadzać z id bloku tool_use
                    "content": "Email zapisany"
                }]
            })
            response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, tools=tools, messages=messages)

    return next(block.text for block in response.content if block.type == "text")

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Krok 3

Nasz pierwszy w historii Agent Loop, zrobiony bez Agent Frameworka!

Zmiany:
1. Zamiast zawsze zakładać, że jest tylko 1 wywołanie narzędzia, iterujemy po narzędziach pętlą for
2. Zmieniono `if finish_reason=="tool_calls"` na `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]  # Gradio może dawać dodatkowe klucze w historii - Anthropic akceptuje tylko role/content
    messages = history + [{"role": "user", "content": message}]
    response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, tools=tools, messages=messages)

    while response.stop_reason == "tool_use":  # pętla trwa, dopóki Claude chce użyć narzędzia
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    email = block.input.get("email")
                    record_email_tool(email)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": "Email zapisany"
                    })
            messages.append({"role": "user", "content": tool_results})  # wszystkie wyniki narzędzi w JEDNEJ wiadomości user - Claude oczekuje ich razem, nie po jednej na wiadomość
            response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, tools=tools, messages=messages)

    return next(block.text for block in response.content if block.type == "text")

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Gratulacje!

Właśnie zaimplementowałeś Asystenta AI z Narzędziami.  
I ręcznie skręciłeś Agent Loop, bez potrzeby Agent Frameworka.  
To wszystko!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">1. Dodaj wiele wywołań LLM! Po tym, jak LLM sformułuje odpowiedź, użyj kolejnego wywołania LLM, żeby ocenić, czy odpowiedź dotyczy ściśle tylko spraw zawodowych.<br/><br/>2. Zastosuj to w swoim biznesie! Zrób Asystenta AI, który potrafi odpowiadać na pytania o Twój obszar biznesowy i użyj narzędzia do zapisywania adresów mailowych osób, które chcą się skontaktować.
            </span>
        </td>
    </tr>
</table>